# Sourcing Desk: step-by-step walkthrough

Run the cells top to bottom. Each step is the same function the pipeline calls, and every call is also
recorded to `runs/<run_id>/<step>__<part>.json` (input, output, API calls, timing, errors).

- Crustdata responses are cached in `.cache/crustdata/`, so re-running a cell costs **0 credits**.
  Set `use_cache=False` to force fresh data.
- Kernel: pick the repo's `.venv` (Python 3.13).

In [ ]:
import json
import pandas as pd
from desk import ROOT, pipeline, steps, view
from desk.crustdata import Crustdata
from desk.trace import Run, list_runs, load_run

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_columns", 30)

req = pipeline.load_request(ROOT / "examples" / "electronics_bom.yaml")
pack = steps.load_pack(req.category)
cd = Crustdata(use_cache=True)       # False = always hit the API (costs credits)
run = Run("notebook")
run.record("00_request", req)
print(run.dir)
req

## Step 1: understand the specs
v0 has no LLM: every BOM line must already carry an exact part number (`mpn`) and maker.

In [ ]:
specs = run.step("01_spec", steps.understand_specs, req)
pd.DataFrame([s.model_dump() for s in specs])

## Step 2: find suppliers, one part
Change `PART` to look at another line: `mcu`, `ldo`, `cap10u`, `usbc`.

In [ ]:
PART = "ldo"
spec = next(s for s in specs if s.line_id == PART)
find = run.step("02_find", steps.find_suppliers, spec, pack, cd, part=PART)
print(find.status.value, "|", find.reason)
view.quotes(find)

Price tiers behind each quote (the pipeline picks the cheapest tier we qualify for at our quantity).

In [ ]:
view.tiers(find)

Other parts seen on the way: different part numbers, or the same part number from another maker (possible clones). Substitute candidates later.

In [ ]:
view.quotes(find, alternatives=True)

Unknown sellers from the open search. Step 3 will check these; `domain` is empty when the link is a Google redirect.

In [ ]:
view.leads(find)

The API calls this step made, with credits and cache hits.

In [ ]:
view.calls(run.records[-1:])

### Drill into a raw page
The exact page text a quote was read from.

In [ ]:
q = next(q for q in find.quotes if q.cache_key and q.status.value == "FOUND")
print(q.vendor, q.url, "| cache_key:", q.cache_key)
view.page_text(q.cache_key, chars=1500, around="Stock")

## Steps 2 to 6 for the whole BOM
Step 3 (check), 4 (buy data) and 6 (order) are stubs; step 5 is a placeholder rule.

In [ ]:
finds = {s.line_id: run.step("02_find", steps.find_suppliers, s, pack, cd, part=s.line_id) for s in specs}
pd.DataFrame([{"part": k, "status": f.status.value, "reason": f.reason} for k, f in finds.items()])

In [ ]:
checks = run.step("03_check", steps.check_suppliers, list(finds.values()))
pd.DataFrame([c.model_dump() for c in checks])

In [ ]:
gaps = run.step("04_buy_data", steps.buy_data, list(finds.values()), req.data_budget_inr)
pd.DataFrame(gaps)

In [ ]:
decisions = {s.line_id: run.step("05_decide", steps.decide, s, finds[s.line_id], checks, part=s.line_id) for s in specs}
pd.DataFrame([{"part": k, "picked": d.picked.vendor if d.picked else None, "total_inr": d.total_inr,
               "reason": d.reason} for k, d in decisions.items()])

In [ ]:
orders = {k: run.step("06_order", steps.place_order, d, run.id, part=k) for k, d in decisions.items()}
pd.DataFrame([o.model_dump() for o in orders.values() if o])

## This run, step by step

In [ ]:
view.steps(run.records)

## Look at any past run
Includes runs from the CLI (`uv run python -m desk run examples/electronics_bom.yaml`).

In [ ]:
[p.name for p in list_runs()[:10]]

In [ ]:
records = load_run()                 # latest run; or load_run("20261122-131235_electronics_bom")
view.steps(records)

In [ ]:
rec = next(r for r in records if r["step"] == "02_find" and r["part"] == "ldo")
print(json.dumps(rec["output"], indent=1, ensure_ascii=False)[:4000])